In [1]:
!pip install requests==2.32.5 \
             opentelemetry-api==1.40.0 \
             opentelemetry-sdk==1.40.0 \
             opentelemetry-exporter-otlp-proto-common==1.40.0 \
             opentelemetry-proto==1.40.0

In [2]:
%pip install -qU langchain langchain-community langchain-google-genai chromadb langgraph

In [3]:
pip install langchain-chroma

In [4]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

First, mount your Google Drive to access files hosted there.

In [5]:
from chromadb.config import Settings
from google.colab import userdata
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

import json
import chromadb

GEMINI_API_KEY=userdata.get('GOOGLE_API_KEY')

In [6]:
import os
from pathlib import Path

if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  file_path = '/content/drive/MyDrive/GenAI/data/dev.json'
  db_path = '/content/drive/MyDrive/GenAI/data/dbs/dev_databases/formula_1/formula_1.sqlite'
  vector_db_path = '/content/drive/MyDrive/GenAI/data/chromadb'
  databases_dir = '/content/drive/MyDrive/GenAI/data/dbs/dev_databases'
else:
  # Define paths for local environment
  # IMPORTANT: Update these paths to match your local setup
  file_path = str(Path('data') / 'dev.json')
  db_path = str(Path('data') / 'formula_1' / 'formula_1.sqlite')
  vector_db_path = str(Path('data') / 'chromadb')
  databases_dir = str(Path('data') / 'dbs')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Once your Drive is mounted, you can specify the path to your data file. For example, if you have a CSV file named `my_data.csv` in your Drive, you can load it into a pandas DataFrame like this:

In [7]:
# IMPORTANT: Replace 'path/to/your/my_data.csv' with the actual path to your file
def load_data(file_path):
  try:
    with open(file_path,"r") as f:
      return json.load(f)
  except FileNotFoundError:
    print(f"❌ Error: File not found at {file_path}")
    return None
  except json.JSONDecodeError as e:
    print(f"❌ Error: Could not decode JSON from {file_path}: {e}")
    return None
  except Exception as e:
    print(f"❌ An unexpected error occurred while loading data from {file_path}: {e}")
    return None

In [8]:
def initialize_gemini_embeddings(api_key: str):
    """Initializes and returns GoogleGenerativeAIEmbeddings."""
    print("✨ Initializing Gemini Embeddings...")
    try:
        return GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=api_key)
    except Exception as e:
        print(f"❌ Error initializing Gemini Embeddings: {e}")
        return None

def get_chroma_client(vector_db_path: str):
    """Initializes and returns a persistent ChromaDB client."""
    print(f"📦 Initializing ChromaDB client at {vector_db_path}...")
    try:
        client = chromadb.PersistentClient(
            path=vector_db_path,
            settings=Settings(anonymized_telemetry=False)
        )
        return client
    except Exception as e:
        print(f"❌ Error initializing ChromaDB client at {vector_db_path}: {e}")
        return None

def get_vector_store(chroma_client, embeddings, collection_name: str = "documents"):
    """
    Initializes and returns a Chroma vector store.
    Deletes existing collection if it exists.
    """
    # print(f"🗑️ Deleting existing '{collection_name}' collection (if any)...")
    print(f"📚 Initializing Chroma vector store with collection '{collection_name}'...")
    vector_store = Chroma(
        client=chroma_client,
        collection_name=collection_name,
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("✅ Chroma vector store initialized.")
    return vector_store

def prepare_documents_from_dev_data(dev_data: list):
    """
    Prepares a list of Document objects from the raw development data.
    """
    print("📄 Preparing documents from development data...")
    documents = []
    for item in dev_data:
        evidences = [e.strip() for e in item['evidence'].split(';') if e.strip()]
        for evidence in evidences:
            doc = Document(
                page_content=evidence,
                metadata={
                    "db_id": item["db_id"]
                }
            )
            documents.append(doc)
    print(f"✅ Prepared {len(documents)} documents.")
    return documents

def populate_vector_store_with_documents(vector_store, documents):
    """Adds documents to the given vector store."""
    print(f"📚 Loading {len(documents)} text-to-SQL examples into vector store...")
    vector_store.add_documents(documents)
    print(f"✅ Added {len(documents)} text-to-SQL examples to the vector store")

def log_database_distribution(dev_data: list):
    """Calculates and prints the distribution of database IDs in the data."""
    db_counts = {}
    for item in dev_data:
        db_id = item['db_id']
        db_counts[db_id] = db_counts.get(db_id, 0) + 1

    print(f"\n📊 Database distribution:")
    for db_id, count in sorted(db_counts.items()):
        print(f"  - {db_id}: {count} examples")

# Main execution flow
dev_data = load_data(file_path)

# Build RAG Workflow

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import List, Dict, Any ,Annotated

from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field
from functools import partial
from langchain_google_genai import ChatGoogleGenerativeAI
import sqlite3

In [10]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage

class RetrievedDocument(BaseModel):
    """Represents a document retrieved from the vector store"""
    content: str = Field(description="The main content/text of the document")
    metadata: Dict[str, Any] = Field(description="Metadata associated with the document")
    score: float = Field(description="Similarity score")

class RAGState(BaseModel):
    """State for the RAG workflow"""
    db_path: str = ""
    sql_result: Dict[str, Any] = Field(default_factory=dict)
    model_name: str = "gemini-flash-latest"  # Updated to a confirmed available model
    messages: Annotated[List[BaseMessage], add_messages] = Field(default_factory=list)
    query: str = ""
    sql_query: str = ""
    reasoning: str = ""
    retrieved_documents: List[RetrievedDocument] = Field(default_factory=list)
    context: str = ""
    final_answer: str = ""
    k: int = 5
    db_id: str = ""

In [11]:
def begin_conversation(state: RAGState) -> Dict[str, Any]:
    # Create the prompt template
    system_prompt = """\
You are a helpful AI assistant that generates SQL queries to answer the user's question.\
"""

    return {
        "messages": [
            SystemMessage(content=system_prompt)
    ]
    }

In [12]:
def process_query(state: RAGState) -> Dict[str, Any]:
    """Process the incoming query and prepare for retrieval"""
    print(f"🔍 Processing query: {state.query}")

    # Add user message if not already present
    messages = state.messages.copy()
    if not any(isinstance(msg, HumanMessage) and msg.content == state.query for msg in messages):
        messages.append(HumanMessage(content=state.query))

    return {
        "messages": messages,
    }

In [13]:
def retrieve_documents(state: RAGState,vector_store) -> Dict[str, Any]:
    """Retrieve relevant documents from ChromaDB, matching the db_id in state."""
    print(f"📚 Retrieving top {state.k} documents for query: {state.query} (db_id: {state.db_id})")

    try:
        # Perform similarity search, filtering by db_id in metadata
        results = vector_store.similarity_search_with_score(
            query=state.query,
            k=state.k,
            filter={"db_id": state.db_id} if state.db_id else None
        )

        # Convert to RetrievedDocument objects
        retrieved_docs = []
        for doc, score in results:
            retrieved_docs.append(RetrievedDocument(
                content=doc.page_content,
                metadata=doc.metadata,
                score=1 - score  # Convert distance to similarity score
            ))

        print(f"✅ Retrieved {len(retrieved_docs)} documents")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"   {i}. Score: {doc.score:.4f} | Content: {doc.content[:100]}...")

        # Create context from retrieved documents
        context_parts = []
        for i, doc in enumerate(retrieved_docs, 1):
            context_parts.append(f"Document {i}:\n{doc.content}")
            if doc.metadata:
                context_parts.append(f"Metadata: {doc.metadata}")
            context_parts.append("")  # Empty line for separation

        context = "\n".join(context_parts)

        return {
            "retrieved_documents": retrieved_docs,
            "context": context,
        }

    except Exception as e:
        print(f"❌ Error retrieving documents: {e}")
        return {
            "retrieved_documents": [],
            "context": "No documents could be retrieved due to an error.",
        }

In [14]:
class SQLOutput(BaseModel):
    reasoning: str = Field(description="think through step by step how to solve the problem.")
    sql_query: str = Field(description="The executable sql query")

    def dict(self):
        return {
            "reasoning": self.reasoning,
            "sql_query": self.sql_query
        }

In [15]:
import sqlite3
from typing import Dict, List, Tuple

class DatabaseSchemaManager:
    """Manages database schema information for a given SQLite database."""

    def __init__(self, db_path: str):
        self.db_path = db_path
        if not self._test_connection():
            raise ValueError(f"Could not connect to database at {db_path}")

    def _test_connection(self) -> bool:
        """Tests if a connection to the database can be established."""
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            return True
        except sqlite3.Error:
            return False
        finally:
            if conn:
                conn.close()

    def _get_cursor(self):
        """Establishes a connection and returns a cursor."""
        conn = sqlite3.connect(self.db_path)
        return conn, conn.cursor()

    def _get_tables(self, cursor) -> List[str]:
        """Retrieves the names of all user-defined tables in the database."""
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
        return [row[0] for row in cursor.fetchall()]

    def _get_columns(self, cursor, table: str) -> List[Tuple[str, str]]:
        """Retrieves column names and types for a given table."""
        cursor.execute(f"PRAGMA table_info({table});")
        return [(row[1], row[2]) for row in cursor.fetchall()]  # (column_name, data_type)

    def _get_foreign_keys(self, cursor, table: str) -> List[Tuple[str, str, str]]:
        """Retrieves foreign key information for a given table."""
        cursor.execute(f"PRAGMA foreign_key_list({table});")
        return [(row[3], row[2], row[4]) for row in cursor.fetchall()]  # (from_col, ref_table, ref_col)

    def describe_database(self) -> str:
        """Generates a human-readable description of the database schema."""
        conn = None
        try:
            conn, cursor = self._get_cursor()
            tables = self._get_tables(cursor)

            description = []
            for table in tables:
                description.append(f"\n🧱 Table: {table}")
                # Columns
                columns = self._get_columns(cursor, table)
                for name, dtype in columns:
                    description.append(f"   🔸 {name} ({dtype})")
                # Foreign keys
                fks = self._get_foreign_keys(cursor, table)
                if fks:
                    description.append("   🔗 Foreign Keys:")
                    for from_col, ref_table, ref_col in fks:
                        description.append(f"      {from_col} \u2192 {ref_table}.{ref_col}")
            return "\n".join(description)
        except sqlite3.Error as e:
            print(f"Error accessing database at {self.db_path}: {e}")
            return f"Error: Could not describe database at {self.db_path} - {e}"
        finally:
            if conn:
                conn.close()

# Helper function for backwards compatibility or direct printing
def print_database_description(db_path: str):
    """Prints the database schema description for a given database path."""
    try:
        manager = DatabaseSchemaManager(db_path)
        print(manager.describe_database())
    except ValueError as e:
        print(e)

def execute_sql_query(db_path: str, query: str) -> List[Dict[str, Any]]:
    """Executes a SQL query against the specified database and returns results as list of dicts."""
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()
        columns = [description[0] for description in cursor.description]
        results = []
        for row in rows:
            results.append(dict(zip(columns, row)))
        return results
    except sqlite3.Error as e:
        print(f"Error executing SQL query: {e}")
        return []
    finally:
        if conn:
            conn.close()

In [16]:
print_database_description(db_path=db_path)


🧱 Table: circuits
   🔸 circuitId (INTEGER)
   🔸 circuitRef (TEXT)
   🔸 name (TEXT)
   🔸 location (TEXT)
   🔸 country (TEXT)
   🔸 lat (REAL)
   🔸 lng (REAL)
   🔸 alt (INTEGER)
   🔸 url (TEXT)

🧱 Table: constructors
   🔸 constructorId (INTEGER)
   🔸 constructorRef (TEXT)
   🔸 name (TEXT)
   🔸 nationality (TEXT)
   🔸 url (TEXT)

🧱 Table: drivers
   🔸 driverId (INTEGER)
   🔸 driverRef (TEXT)
   🔸 number (INTEGER)
   🔸 code (TEXT)
   🔸 forename (TEXT)
   🔸 surname (TEXT)
   🔸 dob (DATE)
   🔸 nationality (TEXT)
   🔸 url (TEXT)

🧱 Table: seasons
   🔸 year (INTEGER)
   🔸 url (TEXT)

🧱 Table: races
   🔸 raceId (INTEGER)
   🔸 year (INTEGER)
   🔸 round (INTEGER)
   🔸 circuitId (INTEGER)
   🔸 name (TEXT)
   🔸 date (DATE)
   🔸 time (TEXT)
   🔸 url (TEXT)
   🔗 Foreign Keys:
      circuitId → circuits.circuitId
      year → seasons.year

🧱 Table: constructorResults
   🔸 constructorResultsId (INTEGER)
   🔸 raceId (INTEGER)
   🔸 constructorId (INTEGER)
   🔸 points (REAL)
   🔸 status (TEXT)
   🔗 Foreig

In [17]:
def generate_sql_query(state: RAGState) -> Dict[str, Any]:
    """Generate sql query using retrieved documents"""
    print(f"🤖 Generating answer using model: {state.model_name}...")

    # Initialize LLM with the updated model name
    llm = ChatGoogleGenerativeAI(
        model=state.model_name,
        temperature=0,
        google_api_key=GEMINI_API_KEY
    ).with_structured_output(SQLOutput)

    manager = DatabaseSchemaManager(state.db_path)
    new_message = HumanMessage(content=f"Question: {state.query}\nContext: {state.context}\nDatabase Description: {manager.describe_database()}")

    messages = [*state.messages, new_message]

    try:
        response = llm.invoke(messages)
        return {
            "messages": [
                *messages,
                AIMessage(content=f'Reasoning: {response.reasoning}\nSQL Query: {response.sql_query}')
            ],
            "sql_query": response.sql_query,
            "reasoning": response.reasoning,
        }
    except Exception as e:
        error_msg = f"Error generating answer: {e}"
        print(f"❌ {error_msg}")
        return {"messages": [*state.messages, AIMessage(content=error_msg)], "final_answer": error_msg}

In [18]:
def run_sql_query(state: RAGState) -> Dict[str, Any]:
    """Execute SQL query on the specified database"""
    import sqlite3
    import os

    print(f"Executing SQL query on database '{state.db_id}': {state.sql_query}")

    try:

        if not os.path.exists(state.db_path):
            error_msg = f"Database {state.db_id} not found at {state.db_path}"
            print(f" {error_msg}")
            return {
                "sql_result": {"error": error_msg},
                "final_answer": f"Error: {error_msg}",
            }

        # Connect to SQLite database
        conn = sqlite3.connect(state.db_path)
        cursor = conn.cursor()

        # Execute the query
        cursor.execute(state.sql_query)

        # Get column names
        columns = (
            [description[0] for description in cursor.description]
            if cursor.description
            else []
        )

        # Fetch results
        results = cursor.fetchall()

        conn.close()

        sql_result = {
            "success": True,
            "columns": columns,
            "data": results,
            "row_count": len(results),
            "sql_query": state.sql_query,
        }

        print(f"SQL query executed successfully! Found {len(results)} row(s)")

        # Format results for final answer
        if results:
            result_text = (
                f"Query executed successfully! Found {len(results)} row(s):\\n\\n"
            )

            # Add column headers
            if columns:
                result_text += " | ".join(columns) + "\\n"
                result_text += "-" * (len(" | ".join(columns))) + "\\n"

            # Add data rows (limit to first 10 for readability)
            for i, row in enumerate(results[:10]):
                result_text += (
                    " | ".join(
                        str(cell) if cell is not None else "NULL" for cell in row
                    )
                    + "\\n"
                )

            if len(results) > 10:
                result_text += f"\\n... and {len(results) - 10} more row(s)"
        else:
            result_text = "Query executed successfully but returned no results."

        print()
        print(result_text)

        return {"sql_result": sql_result, "final_answer": result_text}

    except Exception as e:
        error_msg = f"Error executing SQL query: {str(e)}"
        print(f"{error_msg}")
        return {"sql_result": {"error": error_msg}, "final_answer": error_msg}

In [19]:
from langgraph.graph import StateGraph, END

In [20]:
def create_workflow(vector_store):
    # Create the workflow graph
    workflow = StateGraph(RAGState)

    # Add nodes
    workflow.add_node("begin_conversation", begin_conversation)
    workflow.add_node("process_query", process_query)
    workflow.add_node(
        "retrieve_documents", partial(retrieve_documents, vector_store=vector_store)
    )
    workflow.add_node("generate_sql_query", generate_sql_query)
    workflow.add_node("run_sql_query", run_sql_query)

    # Add edges
    workflow.add_edge("begin_conversation", "process_query")
    workflow.add_edge("process_query", "retrieve_documents")
    workflow.add_edge("retrieve_documents", "generate_sql_query")
    workflow.add_edge("generate_sql_query", "run_sql_query")
    workflow.add_edge("run_sql_query", END)

    # Set entry point
    workflow.set_entry_point("begin_conversation")
    return workflow

In [21]:
embeddings = initialize_gemini_embeddings(GEMINI_API_KEY)
chroma_client = get_chroma_client(vector_db_path)
vector_store = get_vector_store(chroma_client, embeddings)

✨ Initializing Gemini Embeddings...
📦 Initializing ChromaDB client at /content/drive/MyDrive/GenAI/data/chromadb...
📚 Initializing Chroma vector store with collection 'documents'...
✅ Chroma vector store initialized.


In [22]:
workflow = create_workflow(vector_store)
print("Workflow re-compiled with new model!")

Workflow re-compiled with new model!


In [23]:

db_id = dev_data[0]["db_id"]

In [24]:
db_path = os.path.join(databases_dir, db_id, f"{db_id}.sqlite")
print(db_path)

/content/drive/MyDrive/GenAI/data/dbs/dev_databases/california_schools/california_schools.sqlite


In [25]:
compiled_workflow = workflow.compile()

In [26]:
result = compiled_workflow.invoke(
        {"query": dev_data[0]["question"], "db_id": db_id, "db_path": db_path}
    )

🔍 Processing query: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
📚 Retrieving top 5 documents for query: What is the highest eligible free rate for K-12 students in the schools in Alameda County? (db_id: california_schools)
✅ Retrieved 5 documents
   1. Score: 0.7457 | Content: Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`...
   2. Score: 0.7457 | Content: Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`...
   3. Score: 0.7340 | Content: Eligible free rates for students aged 5-17 = `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)...
   4. Score: 0.7340 | Content: Eligible free rates for students aged 5-17 = `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)...
   5. Score: 0.7340 | Content: Eligible free rates for students aged 5-17 = `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)...
🤖 Generating answer using model: gemini-flash-latest...
Executing SQL query on 

In [29]:
if chroma_client:
    print("Closing ChromaDB client...")
    chroma_client.close()
    print("✅ ChromaDB client closed.")

Closing ChromaDB client...
✅ ChromaDB client closed.
